# LLM Feature Keşif Döngüsü (LLM Mesh — kütüphanesiz)

`HAVALE_EFT_NTT_prepared` üzerinde qwen36-27b ile otomatik **feature** üretimi.
Kolon haritası `COMPACT_JSON` folder'ındaki `degiskenler_compact.json` dosyasından okunur.

Model her turda ya `AILE:` ile kolon açıklaması ister, ya ```python kodu çalıştırır,
ya da kanıtı yeterliyse `FEATURE / AD / FORMUL / GEREKCE` yazar.
**DEMİR KURAL:** GEREKCE'deki her sayı bu oturumda çalıştırılan kodun print çıktısından gelir.
Keşif TRAIN alt kümesinde yapılır; holdout dokunulmaz.

Recipe girdileri: HAVALE_EFT_NTT_prepared + DEGISKENLER + COMPACT_JSON (folder).
Çıktı: FEATURE_ONERILERI dataset'i.


## 1. Kütüphaneler

In [ ]:
import io
import re
import json
import contextlib
import traceback
from collections import defaultdict

import dataiku
import numpy as np
import pandas as pd


## 2. Konfigürasyon

In [ ]:
# --- Veri ---
DATA_DATASET   = "HAVALE_EFT_NTT_prepared"
DICT_DATASET   = "DEGISKENLER"          # orijinal sözlük (AILE açma için)
COMPACT_FOLDER = "COMPACT_JSON"         # compact JSON'un durduğu managed folder
COMPACT_FILE   = "degiskenler_compact.json"
NAME_COL, DESC_COL = "SM_ID", "aciklama"
TARGET_COL     = "TGT_CS_ALL_TKP_ST_5M90"   # 1 = FPD, 0 = sağlam

# --- LLM ---
LLM_ID = "openai:dataiku-qwen36-27b:qwen36-27b"

# --- Koşu bütçesi ---
N_TURNS           = 12     # toplam LLM turu
N_FEATURES        = 5      # hedef feature sayısı
TRAIN_FRAC        = 0.7    # keşif bu alt kümede; holdout dokunulmaz
SEED              = 42
MAX_OUTPUT_CHARS  = 4000   # tek script çıktısından prompta taşınacak azami
MAX_HISTORY_CHARS = 24000  # geçmiş bütçesi

# --- Çıktı ---
OUT_DATASET = "FEATURE_ONERILERI"


## 3. LLM bağlantısı (LLM Mesh)

In [ ]:
project = dataiku.api_client().get_default_project()
llm = project.get_llm(LLM_ID)


def ask_llm(system: str, user: str) -> str:
    comp = llm.new_completion()
    try:
        comp.with_message(system, role="system")
        comp.with_message(user, role="user")
    except TypeError:  # eski API'de role parametresi yoksa
        comp = llm.new_completion()
        comp.with_message(system + "\n\n" + user)
    resp = comp.execute()
    if not getattr(resp, "success", True):
        raise RuntimeError(f"LLM çağrısı başarısız: {getattr(resp, 'errorMessage', resp)}")
    return resp.text


print("LLM hazır:", LLM_ID)


## 4. Compact'ı folder'dan oku + AILE açma haritası
Nav = kompakt JSON (her turda promptta). `open_families` istenen ailelerin
kolonlarını DEGISKENLER'deki **orijinal** açıklamalarıyla döndürür.

In [ ]:
folder = dataiku.Folder(COMPACT_FOLDER)
with folder.get_download_stream(COMPACT_FILE) as f:
    compact = json.loads(f.read().decode("utf-8"))

NAV_JSON = json.dumps(compact, ensure_ascii=False, separators=(",", ":"))
print(f"Compact yüklendi: {len(compact['aileler'])} aile, "
      f"nav {len(NAV_JSON)} karakter (~{len(NAV_JSON)//3} token)")

NEW_SUFFIXES = sorted({
    "AMT_PER_DAY", "CNT_PER_DAY", "AMT_AVG", "AMT_MAX",
    "HS_CNT", "HS_AMT", "BNK_ADT", "CNT", "AMT",
    "HS_CNT_SHR", "HS_AMT_SHR", "CNT_SHR", "AMT_SHR",
    "CNT_RATIO", "AMT_RATIO", "AMT_STD", "CV",
    "TMSNCFRST", "TMSNCLST",
}, key=len, reverse=True)


def family_of(col: str):
    parts = col.split("_")
    if parts[0] in ("OUT", "IN") and len(parts) >= 3:
        return "_".join(parts[:2])
    for suf in NEW_SUFFIXES:
        if col.endswith("_" + suf):
            return col[: -(len(suf) + 1)]
    return None


dict_df = dataiku.Dataset(DICT_DATASET).get_dataframe()
dict_df.columns = [c.strip() for c in dict_df.columns]
FAMILY_COLUMNS = defaultdict(list)
for _, r in dict_df.iterrows():
    col = str(r[NAME_COL]).strip()
    fam = family_of(col)
    if fam:
        FAMILY_COLUMNS[fam].append((col, str(r[DESC_COL]).strip()))
print(f"Aile haritası: {len(FAMILY_COLUMNS)} aile")


def open_families(fams) -> str:
    out = []
    for f in fams:
        f = f.strip().strip('`"\'')
        if f in FAMILY_COLUMNS:
            out.append(f"### {f}")
            out += [f"- `{c}`: {d}" for c, d in FAMILY_COLUMNS[f]]
        else:
            out.append(f"### {f}\n- (böyle bir aile yok — JSON'daki aile anahtarlarını kullan)")
    return "\n".join(out)


## 5. Veri: yükle, train/holdout ayır, base rate

In [ ]:
main_df = dataiku.Dataset(DATA_DATASET).get_dataframe(infer_with_pandas=True)
assert TARGET_COL in main_df.columns, f"{TARGET_COL} veride yok!"

rng = np.random.RandomState(SEED)
mask = rng.rand(len(main_df)) < TRAIN_FRAC
df = main_df[mask].reset_index(drop=True)        # keşif verisi (LLM sadece bunu görür)
holdout = main_df[~mask].reset_index(drop=True)  # dokunulmaz

BASE_RATE = float(df[TARGET_COL].mean())
print(f"Toplam: {len(main_df):,} | Train: {len(df):,} | Holdout: {len(holdout):,}")
print(f"Train base rate (FPD): {BASE_RATE:.2%}")
del main_df


## 6. Kod çalıştırıcı
Modelin analiz kodu kontrollü çalışır (`df`, `pd`, `np`, `TARGET`, `BASE_RATE` hazır);
print çıktısı yakalanıp modele geri verilir, hata kısaltılıp aynen iletilir.

In [ ]:
def run_code(code_str: str) -> str:
    buf = io.StringIO()
    env = {"df": df, "pd": pd, "np": np, "TARGET": TARGET_COL, "BASE_RATE": BASE_RATE}
    try:
        with contextlib.redirect_stdout(buf):
            exec(code_str, env)
        out = buf.getvalue().strip() or "(kod çalıştı ama print yok — sayıları print et!)"
    except Exception:
        out = "HATA:\n" + traceback.format_exc(limit=3)
    if len(out) > MAX_OUTPUT_CHARS:
        out = out[:MAX_OUTPUT_CHARS] + "\n... (çıktı kırpıldı — daha az satır print et)"
    return out


## 7. Prompt + keşif döngüsü

In [ ]:
SYSTEM = f"""Kıdemli kredi riski veri analistisin. Amaç: FPD'yi (`{TARGET_COL}`=1,
base rate {BASE_RATE:.2%}) ayrıştıran YENİ feature'lar üretmek.

Sana her turda kolon haritası JSON verilecek: tam kolon adı = 'aileler' anahtarı + '_' + metrik;
anlamlar 'lejant' bölümünde.

Her cevabında ŞU ÜÇÜNDEN TAM OLARAK BİRİNİ yap:
1) `AILE: aile1, aile2` yaz (en fazla 4) -> o ailelerin kolon açıklamaları gelir.
2) TEK bir ```python bloğu yaz. df (train), pd, np, TARGET, BASE_RATE hazır.
   Segment bad-rate, base rate farkı (pp-gap), lift ve segment n'ini PRINT ET.
3) Kanıt yeterliyse feature yaz, TAM bu şablonla:
FEATURE:
AD: <snake_case ad>
FORMUL: df['<ad>'] = <pandas ifadesi; oran/etkileşim/eşik tercih; bölmede +1e-9>
GEREKCE: <1-2 cümle + BU oturumdaki kod ÇIKTISINDAN birebir sayılar (rate, lift, n)>

DEMİR KURAL: yazdığın her sayı bu oturumda çalıştırdığın kodun print çıktısından
birebir kopya olmalı; kafadan hesap ve tahmin yasak.
SIFIR ŞİŞKİNLİK: çoğu kolon işlemsiz müşteride 0'dır; 'sağlam medyanı 0' eşik olamaz;
sıfır-dışı alt kümede analiz et, iki grubun sıfır payını da print et.
n < 200 segmentlere güvenme. Kolon adından emin değilsen önce AILE aç. Türkçe yaz."""


def build_user(opened, history, turn, n_feat):
    parts = ["## KOLON HARİTASI (JSON)", NAV_JSON]
    if opened:
        parts += ["", "## AÇILAN AİLELER (orijinal açıklamalar)", opened]
    parts += ["", "## GEÇMİŞ (kodların ve GERÇEK çıktıları)",
              history if history else "(henüz yok)",
              "", f"## TUR {turn}/{N_TURNS} — Üretilen feature: {n_feat}/{N_FEATURES}",
              "Sıradaki TEK adımını seç: AILE / ```python / FEATURE."]
    return "\n".join(parts)


history, opened_block, features = "", "", []

for turn in range(1, N_TURNS + 1):
    reply = ask_llm(SYSTEM, build_user(opened_block, history, turn, len(features)))
    opened_block = ""

    m_feat = re.search(r"FEATURE:(.*)", reply, re.S)
    m_code = re.search(r"```python(.*?)```", reply, re.S)
    m_aile = re.search(r"^AILE:\s*(.+)$", reply, re.M)

    if m_feat:
        feat_text = "FEATURE:\n" + m_feat.group(1).strip()
        features.append(feat_text)
        history += f"\n[TUR {turn}] Feature #{len(features)} kaydedildi.\n"
        print(f"[TUR {turn}] ✅ FEATURE #{len(features)}")
        if len(features) >= N_FEATURES:
            print("Hedefe ulaşıldı.")
            break
    elif m_code:
        out = run_code(m_code.group(1))
        history += f"\n[TUR {turn}] --- KOD ---\n{m_code.group(1).strip()}\n--- ÇIKTI ---\n{out}\n"
        print(f"[TUR {turn}] 🐍 kod çalıştı ({len(out)} kr çıktı)")
    elif m_aile:
        fams = [f.strip() for f in m_aile.group(1).split(",") if f.strip()][:4]
        opened_block = open_families(fams)
        history += f"\n[TUR {turn}] Aile açıldı: {', '.join(fams)}\n"
        print(f"[TUR {turn}] 📂 aile açıldı: {fams}")
    else:
        history += f"\n[TUR {turn}] (Geçersiz format — AILE / ```python / FEATURE bekleniyor)\n"
        print(f"[TUR {turn}] ⚠️ geçersiz cevap")

    if len(history) > MAX_HISTORY_CHARS:
        history = "...(eski turlar kırpıldı)...\n" + history[-MAX_HISTORY_CHARS:]

print(f"\nDöngü bitti: {len(features)} feature, {turn} tur.")


## 8. Final: feature'ları ayrıştır ve dataset'e yaz
Şablon alanları ayrıştırılır; ayrışamayan feature ham metniyle korunur (kayıp olmaz).

In [ ]:
def parse_feat(text: str) -> dict:
    def sec(name):
        m = re.search(rf"^{name}:\s*(.+?)(?=^\w+:|\Z)", text, re.S | re.M)
        return " ".join(m.group(1).split()) if m else ""
    return {"ad": sec("AD"), "formul": sec("FORMUL"), "gerekce": sec("GEREKCE"), "ham": text}


out = pd.DataFrame([parse_feat(f) for f in features],
                   columns=["ad", "formul", "gerekce", "ham"])
out.insert(0, "feature_id", [f"F{i+1:02d}" for i in range(len(out))])

print(out[["feature_id", "ad", "formul"]].to_string(index=False))
dataiku.Dataset(OUT_DATASET).write_with_schema(out)
print(f"\n{OUT_DATASET} yazıldı ✓ ({len(out)} feature)")
